In [1]:
FORCE_RELOAD = False
DATASET_NAME = 'DiscoveryWorld'
DATASET_VERSION = 'v2'
DELETE_IF_EXISTS_DATASET = True


In [2]:
from a4ai_project.dataset_manager import datasetManager

df= datasetManager.load_dataset(FORCE_RELOAD)
df

pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
Loading dataset: dataset.parquet


,id,scenarioName,difficultyStr,maxAgents,taskName,taskDescription,maxScore,scorecard
0,0,Tutorial,Normal,1,TutorialTask,Where am I?! You just wake up in a strange pla...,3,"[{'name': 'Talk to elder', 'description': 'The..."
1,1,Combinatorial Chemistry,Easy,1,RustedKeyTaskEasy,You were venturing into the wilderness of Plan...,12,"[{'name': 'Take key', 'description': 'The key ..."
2,2,Combinatorial Chemistry,Normal,1,RustedKeyTaskNormal,You were venturing into the wilderness of Plan...,12,"[{'name': 'Take key', 'description': 'The key ..."
3,3,Combinatorial Chemistry,Challenge,1,RustedKeyTaskChallenge,You were venturing into the wilderness of Plan...,12,"[{'name': 'Take key', 'description': 'The key ..."
4,4,Archaeology Dating,Easy,1,ArchaeologyDigTaskEasy,You are in archaeological dig store room on Pl...,6,"[{'name': 'Take radiocarbon meter', 'descripti..."
5,5,Archaeology Dating,Normal,5,ArchaeologyDigTaskNormal,You are on an archaeological dig on Planet X. ...,10,"[{'name': 'Take radiocarbon meter', 'descripti..."
6,6,Archaeology Dating,Challenge,5,ArchaeologyDigTaskGenericRadioisotope,You are on an archaeological dig on Planet X. ...,13,"[{'name': 'Take radioisotope meter', 'descript..."
7,7,Plant Nutrients,Easy,1,SoilNutrientTaskEasy,You are at a botanical research station on Pla...,4,"[{'name': 'Soil Nutrient Meter Present', 'desc..."
8,8,Plant Nutrients,Normal,5,SoilNutrientTaskNormal,You are at a botanical research station on Pla...,9,"[{'name': 'Soil Nutrient Meter Present', 'desc..."
9,9,Plant Nutrients,Challenge,5,SoilNutrientTaskChallenge,You are at a botanical research station on Pla...,9,"[{'name': 'Soil Nutrient Meter Present', 'desc..."


In [3]:
df = df[
    (df["scenarioName"] == "Reactor Lab")
    & (df["difficultyStr"] == "Easy")
]
df
# print(df.iloc[10]['taskDescription'])

print(df.iloc[0]['taskDescription'])

You are at the Quantum Reactor Lab on Planet X. Quantum Crystals offer the potential to generate a great deal of power, but require their respective crystal reactors to be tuned to a specific frequency that appears unique for each crystal. Through great effort, a previous research scientist manually stumbled upon the correct frequencies for Crystal 1 and Crystal 2, which are in their respective reactors. Your task is to use science to figure out the correct frequency for Crystal 3, set it's reactor to the appropriate frequency, and install the crystal. Once all three crystals are installed, the reactor will be able to generate a great deal of power. To support your task, a scientific instrument is available in the lab. 


In [4]:
df.columns

Index(['id', 'scenarioName', 'difficultyStr', 'maxAgents', 'taskName',
       'taskDescription', 'maxScore', 'scorecard'],
      dtype='str')

In [5]:
from langsmith import Client
from langsmith.schemas import Dataset
from langsmith.utils import LangSmithConflictError
from pandas import DataFrame

client = Client()

def create_examples(dataset: Dataset, df: DataFrame,):
    # if PROTECT_EXPERIMENTS:
    #     print("PROTECT_EXPERIMENTS enabled, skipping...")
    #     return
    
    example_ids = []
    for _,row in df.iterrows():
        example = client.create_example(
            dataset_id=dataset.id,            
            inputs= {
                "id": row["id"],
                "scenarioName": row["scenarioName"],
                "difficultyStr": row["difficultyStr"],
                # "seed": int(row["seed"]),
                "maxAgents": row["maxAgents"],
                "taskName": row["taskName"],
                "taskDescription": row["taskDescription"],
                "scorecard": row["scorecard"]                
            },
            outputs= {
                "maxScore": row["maxScore"]                         
            },
            # metadata={
            #     "category": row["category"],
            #     "intent": row["intent"],
            # }
        )

        example_ids.append(example.id)

def create_dataset(name, description, delete_if_exists= False):
    try:        
        if client.has_dataset(dataset_name=name):
            if delete_if_exists:
                print(f"Dataset '{name}' already exists. Deleting existing dataset...")
                client.delete_dataset(dataset_name=name)
            else:
                print(f"Dataset '{name}' already exists. Skipping and retrieving from Langsmith...")
                dataset = client.read_dataset(dataset_name=name)
                examples = client.list_examples(dataset_id=dataset.id)
                examples = list(examples)
                return dataset, examples

        dataset = client.create_dataset(dataset_name = name, description = description)
        print(f"Dataset '{name}' created successfully.")
        
        print(f"Creating examples: {len(df)}")
        create_examples(dataset, df)
        
        examples = client.list_examples(dataset_id=dataset.id)
        examples = list(examples)
        return dataset, examples    
    except LangSmithConflictError as e:
        print(f"Error on creating Dataset '{name}': {str(e)}")  

dataset_name = f'{DATASET_NAME} - {DATASET_VERSION}'

dataset, examples = create_dataset(
    name = dataset_name,
    description=dataset_name,
    delete_if_exists= DELETE_IF_EXISTS_DATASET
)

Dataset 'DiscoveryWorld - v2' already exists. Deleting existing dataset...
Dataset 'DiscoveryWorld - v2' created successfully.
Creating examples: 1


In [6]:
from dataclasses import dataclass, field
from langchain_core.language_models.chat_models import BaseChatModel
from a4ai_project.agent_runner import AgentRunner
from a4ai_project.agents.AgentsFactory import AgentType

@dataclass
class AgentConfiguration():
    agent_type: AgentType
    thread_id: int
    model: BaseChatModel | None = field(default=None, repr=False)
    num_agents: int = 1
        
def target_function(inputs:dict, agent_config: AgentConfiguration, max_num_steps: int) -> dict:    
    runner = AgentRunner()
    run = runner.runAgent_Agentic if agent_config.agent_type == AgentType.AgenticPlayer else runner.runAgent
    result = run( 
        agent = agent_config.agent_type, 
        scenarioName= inputs["scenarioName"],
        difficultyStr=inputs["difficultyStr"],
        seed=int(inputs.get("seed", 0)),
        numSteps=max_num_steps,
        model=agent_config.model,
        numUserAgents=agent_config.num_agents,
        threadId=agent_config.thread_id,
    )    
    return {
        "agent_result": result
    }
        
def agent_evaluator(inputs: dict, # what was passed into your target function (e.g., the question text).
                    outputs:dict, # what your target function returned (e.g., the model’s answer).
                    reference_outputs:dict # the ground truth answers you attached to each dataset example 
                    ) -> dict:
    result = outputs["agent_result"]
    print(f"agent_result: {result}") 
    return {
        "results": [
            {"key": "completedSuccessfully", "score": bool(result["completedSuccessfully"])},
            {"key": "totalScore", "score": result["totalScore"]},
            {"key": "finalNormalizedScore", "score": result["finalNormalizedScore"]},
            {"key": "stepsTaken", "score": result["stepsTaken"]},
            {"key": "actionsSucceeded", "score": result["actionsSucceeded"]},
            {"key": "actionsFailed", "score": result["actionsFailed"]},
            {"key": "actionsSuccessRate", "score": result["actionsSuccessRate"]},
            {"key": "totalTimeTaken", "score": result["totalTimeTaken"]},
            {"key": "stepsPerSecond", "score": result["stepsPerSecond"]},
        ]
    }
       
def evaluate_agent(agent_config: AgentConfiguration, max_num_steps: int):
    result = client.evaluate(                                 
        lambda inputs:  target_function(inputs, agent_config, max_num_steps),        
        data = examples,
        evaluators = [agent_evaluator],
        experiment_prefix=f"wd-{agent_config.agent_type.value}",
        metadata={
            # "stage": stage,    
            "agent": agent_config.agent_type,       
            "num_agents": agent_config.num_agents,
            "thread_id": agent_config.thread_id,
            # "judge_model": judge_model_name,
            # "training_mode": training_mode,
            # "agentic_memory": agentic_memory_mode,
            # "split": split["name"],
            "n_examples": len(df)
        },                
    )
    return result


c:\dev\AI\Dundalk\advanced_agentic\project_code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:780: ArbitraryTypeWarning: <module 'discoveryworld.DiscoveryWorldAPI' from 'c:\\dev\\AI\\Dundalk\\advanced_agentic\\project_code\\.venv\\Lib\\site-packages\\discoveryworld\\DiscoveryWorldAPI.py'> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warnings.warn(
c:\dev\AI\Dundalk\advanced_agentic\project_code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:780: ArbitraryTypeWarning: <module 'discoveryworld.DiscoveryWorldAPI' from 'c:\\dev\\AI\\Dundalk\\advanced_agentic\\project_code\\.venv\\Lib\\site-packages\\discoveryworld\\DiscoveryWorldAPI.py'> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation si

In [7]:
from a4ai_project.llm import get_llm

model = get_llm("gpt-6-luna", reasoning_effort="medium", reasoning_summary="auto")
AgentConfiguration(agent_type=AgentType.LLMAgent, model=model, num_agents=1, thread_id=2)

AgentConfiguration(agent_type=<AgentType.LLMAgent: 'LLMAgent'>, thread_id=2, num_agents=1)

In [8]:
baseline_model = get_llm("gpt-6-luna", reasoning_effort="medium", reasoning_summary="auto")
agentic_model = get_llm("gpt-6-luna", reasoning_effort="medium", reasoning_summary="auto")
max_num_steps = 50

agent_configurations =[
    # AgentConfiguration(agent_type=AgentType.Random, thread_id=1),
    # AgentConfiguration(agent_type=AgentType.LLMAgent, model=baseline_model, num_agents=1, thread_id=2),
    # AgentConfiguration(agent_type=AgentType.LLMAgent, model=get_llm("gpt-6-astra"), num_agents=1, thread_id=3),
    AgentConfiguration(agent_type=AgentType.LLMAgent, model=baseline_model, num_agents=2, thread_id=4),
    # AgentConfiguration(agent_type=AgentType.AgenticPlayer, model=agentic_model, num_agents=1, thread_id=5),
]

def evaluation_pipeline(agent_configurations, max_num_steps):
    results = []
    for config in agent_configurations:
        result = evaluate_agent(config, max_num_steps)
        results.append(result)
        
results = evaluation_pipeline(agent_configurations, max_num_steps)        
results

View the evaluation results for experiment: 'wd-LLMAgent-eb766855' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/581509a9-9bee-42c0-85c0-9bd8c7d3d7f4/compare?selectedSessions=b847db1a-27d5-47ba-8916-bd47d92c1ec0


Running agent 'LLMAgent' on scenario 'Reactor Lab' with difficulty 'Easy' for 50 steps, seed 0, thread ID 4, agents 2.
Loading sprites from index file: c:\dev\AI\Dundalk\advanced_agentic\project_code\.venv\Lib\site-packages\discoveryworld\assets\spriteIndex.json
	Loaded 36 sprites from pixymoon/CuteRPG_Interior/32x32/CuteRPG_Interior_custom.png.
	Loaded 22 sprites from pixymoon/CuteRPG_Houses/32x32/CuteRPG_Houses_A-modified1.png.
	Loaded 30 sprites from pixymoon/CuteRPG_Desert/32x32/CuteRPG_Desert_Outside_modified.png.
	Loaded 1 sprites from pixymoon/CuteRPG_Desert/32x32/CuteRPG_Desert_Inside.png.
	Loaded 9 sprites from pixymoon/CuteRPG_Village/32x32/CuteRPG_Village_House.png.
	Loaded 9 sprites from pixymoon/CuteRPG_Houses/32x32/CuteRPG_Ho